## Biweekly 20.01

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.io as pio
from scipy import stats
from scipy.stats import chi2_contingency
from typing import Callable

def _resolve_project_root() -> Path:
    """locate project root containing config.py."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'config.py').exists():
            print(candidate)
            return candidate
    raise FileNotFoundError('config.py not found in cwd or parents')


PROJECT_ROOT = _resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from config import GENE_PATHS, SOURCE_PALETTE, VARIANT_PATHS

/Users/markus/in-silico-vg-analysis


In [2]:
def _dedup_scores_by_variant(
    path: str | Path,
    label: str,
    columns: list[str] | None = None
) -> pl.DataFrame:
    """deduplicate scores by variant, keeping max absolute score.
    
    args:
        path (str | Path): parquet file path.
        label (str): dataset label for logging.
        columns (list[str] | None): additional columns to keep (e.g. ['AF'], ['perm_AF']).
    
    returns:
        pl.DataFrame: deduplicated variant table with gene_id, variant_id, raw_score, and requested columns.
    """
    df = pl.read_parquet(path)
    print(f"  loaded {label}: {df.shape}")
    
    # required columns
    required = ['variant_id', 'gene_id', 'raw_score']
    if columns:
        required.extend(columns)
    
    # select needed columns
    df = df.select(required)
    
    # deduplicate by keeping variant with max absolute score
    dedup = (
        df
        .with_columns(pl.col('raw_score').abs().alias('_abs_score'))
        .sort('_abs_score', descending=True)
        .unique(subset=['variant_id'], keep='first')
        .drop('_abs_score')
    )
    
    print(f"  after dedup: {dedup.shape}")
    return dedup

In [3]:
# DEBUG CELL: deduplicate and downsample variants for both datasets
from pathlib import Path
import polars as pl
import pandas as pd
import numpy as np

print("=== DEDUPLICATION AND DOWNSAMPLING ===")
print("PROJECT ROOT:", Path.cwd())

# configuration for both datasets
datasets_config = {
    'clingen': {
        'real_var': VARIANT_PATHS['clingen'],
        'null_var': VARIANT_PATHS['clingen_null'],
        'real_gene': GENE_PATHS['clingen'],
    },
    'background': {
        'real_var': VARIANT_PATHS['background'],
        'null_var': VARIANT_PATHS['background_null'],
        'real_gene': GENE_PATHS['background'],
    },
}

print("\n=== INPUT FILES ===")
for dataset_name, config in datasets_config.items():
    print(f"\n{dataset_name.upper()}:")
    print(f"  real_var:  {config['real_var']}")
    print(f"  null_var:  {config['null_var']}")
    print(f"  real_gene: {config['real_gene']}")

# storage for final dataframes
final_dataframes = {}

# helper function for downsampling
def downsample_null_to_real(null_df: pl.DataFrame, real_counts_df: pl.DataFrame, seed: int = 42) -> pl.DataFrame:
    """downsample null variants to match real variant counts per gene."""
    n_map = real_counts_df.select(["gene_id", "n_variants"]).to_pandas().set_index("gene_id")["n_variants"].to_dict()
    pdf = null_df.to_pandas()
    sampled_parts = []
    
    for gene, group in pdf.groupby("gene_id"):
        n_req = int(n_map.get(gene, 0))
        if n_req <= 0:
            continue
        avail = len(group)
        if avail == 0:
            continue
        if avail <= n_req:
            sampled_parts.append(group)
        else:
            taken = group.sample(n=n_req, random_state=seed)
            sampled_parts.append(taken)
    
    if sampled_parts:
        sampled_pdf = pd.concat(sampled_parts, axis=0)
        return pl.from_pandas(sampled_pdf)
    else:
        return pl.DataFrame(schema=null_df.schema)

# process both datasets
for dataset_name, config in datasets_config.items():
    print(f"\n{'='*60}")
    print(f"PROCESSING: {dataset_name.upper()}")
    print(f"{'='*60}")
    
    # load gene table
    real_gene = pl.read_parquet(config['real_gene'])
    print(f"\nreal genes: {real_gene.shape[0]} genes, {real_gene['n_variants'].sum()} total variants")
    print(f"mean: {real_gene['n_variants'].mean():.1f}, median: {real_gene['n_variants'].median():.0f}")
    
    # deduplicate real variants (keep AF column)
    print(f"deduplicating real variants...")
    real_dedup = _dedup_scores_by_variant(config['real_var'], label=f"{dataset_name}_real", columns=['AF'])
    print(f"real dedup: {real_dedup.shape}")
    
    # deduplicate null variants (keep perm_AF column)
    print(f"deduplicating null variants...")
    null_dedup = _dedup_scores_by_variant(config['null_var'], label=f"{dataset_name}_null", columns=['perm_AF'])
    print(f"null dedup: {null_dedup.shape}")
    
    # variant availability check
    null_counts = null_dedup.group_by("gene_id").agg(pl.len().alias("n_null_available"))
    cmp = (
        real_gene.select(["gene_id", "n_variants"])
        .join(null_counts, on="gene_id", how="left")
        .with_columns(pl.col("n_null_available").fill_null(0).cast(pl.Int64))
    )
    
    capped = cmp.filter(pl.col("n_null_available") < pl.col("n_variants"))
    if capped.height > 0:
        deficit = (capped['n_variants'] - capped['n_null_available']).sum()
        print(f"⚠️  {capped.height} genes capped (total deficit: {deficit} variants)")
    else:
        print(f"✓ all genes have sufficient null variants")
    
    # downsample null to match real counts
    print(f"downsampling null variants to match real counts...")
    null_sampled = downsample_null_to_real(null_dedup, real_gene, seed=42)
    print(f"null sampled: {null_sampled.shape}")
    
    # store final dataframes
    final_dataframes[f'{dataset_name}_real_dedup'] = real_dedup
    final_dataframes[f'{dataset_name}_null_sampled'] = null_sampled

# extract final dataframes as variables
print(f"\n{'='*60}")
print("FINAL DATAFRAMES")
print(f"{'='*60}")

clingen_real_dedup = final_dataframes['clingen_real_dedup']
clingen_null_sampled = final_dataframes['clingen_null_sampled']
background_real_dedup = final_dataframes['background_real_dedup']
background_null_sampled = final_dataframes['background_null_sampled']

print(f"\nclingen_real_dedup: {clingen_real_dedup.shape}")
print(f"clingen_null_sampled: {clingen_null_sampled.shape}")
print(f"background_real_dedup: {background_real_dedup.shape}")
print(f"background_null_sampled: {background_null_sampled.shape}")

print("\n✓ all dataframes ready for analysis")
print("\n=== Gene Overlap ===")
real_genes = real_gene.select(pl.col("gene_id").unique())
null_genes = null_dedup.select(pl.col("gene_id").unique())
overlap = real_genes.join(null_genes, on="gene_id", how="inner").height
print(f"unique real genes: {real_genes.height}")
print(f"unique null genes: {null_genes.height}")
print(f"gene overlap: {overlap}")

# 4) Variant availability per gene
print("\n=== Variant Availability ===")
null_counts = null_dedup.group_by("gene_id").agg(pl.len().alias("n_null_available"))
print(f"null variants per gene stats:")
print(null_counts.select('n_null_available').describe())

cmp = (
    real_gene.select(["gene_id", "n_variants"])
    .join(null_counts, on="gene_id", how="left")
    .with_columns(pl.col("n_null_available").fill_null(0).cast(pl.Int64))
)

missing = cmp.filter(pl.col("n_null_available") == 0)
print(f"\nreal genes missing from null: {missing.height}")
if missing.height > 0:
    print(missing.select(["gene_id", "n_variants"]).head(10))

# 5) Genes capped (insufficient null variants)
print("\n=== Capped Genes (null < requested) ===")
capped = cmp.filter(pl.col("n_null_available") < pl.col("n_variants"))
print(f"genes capped: {capped.height}")
if capped.height > 0:
    capped_detail = (
        capped.with_columns((pl.col("n_variants") - pl.col("n_null_available")).alias("deficit"))
        .sort("deficit", descending=True)
        .select(["gene_id", "n_variants", "n_null_available", "deficit"])
    )
    print(f"total deficit (missing variants): {capped_detail['deficit'].sum()}")
    print(capped_detail.head(10))

# 6) Genes with surplus null variants
print("\n=== Surplus Genes (null > requested) ===")
surplus = cmp.filter(pl.col("n_null_available") > pl.col("n_variants"))
print(f"genes with surplus: {surplus.height}")
if surplus.height > 0:
    surplus_detail = (
        surplus.with_columns((pl.col("n_null_available") - pl.col("n_variants")).alias("surplus"))
        .sort("surplus", descending=True)
        .select(["gene_id", "n_variants", "n_null_available", "surplus"])
    )
    print(f"total surplus (extra variants available): {surplus_detail['surplus'].sum()}")
    print("top 10 largest surpluses:")
    print(surplus_detail.head(10).to_pandas())

# 7) Sample variants from problematic genes
print("\n=== Sample Variants from Top Capped Genes ===")
def sample_rows_for_gene(gene_id, n=5):
    print(f"\nsample rows for gene {gene_id}:")
    print(null_dedup.filter(pl.col("gene_id") == gene_id).head(n).to_pandas())

example_genes = []
if capped.height > 0:
    example_genes = capped.sort((pl.col("n_variants") - pl.col("n_null_available")), descending=True).select("gene_id").head(3).to_pandas()["gene_id"].tolist()
elif missing.height > 0:
    example_genes = missing.select("gene_id").head(3).to_pandas()["gene_id"].tolist()
else:
    example_genes = surplus.sort("n_null_available", descending=True).select("gene_id").head(3).to_pandas()["gene_id"].tolist()

for g in example_genes:
    sample_rows_for_gene(g)

# 8) Downsampling with verbose tracking
print("\n=== Downsampling Null Variants ===")
def downsample_verbose(null_df: pl.DataFrame, real_counts_df: pl.DataFrame, seed=42):
    """Return (sampled_pl_df, report_df) where report_df lists per-gene requested, available, taken."""
    # pandas approach for sampling like previous implementation, but produce report
    n_map = real_counts_df.select(["gene_id", "n_variants"]).to_pandas().set_index("gene_id")["n_variants"].to_dict()
    pdf = null_df.to_pandas()
    reports = []
    sampled_parts = []

    rng = np.random.RandomState(seed)

    # groupby gene in pandas for deterministic sampling
    for gene, group in pdf.groupby("gene_id"):
        n_req = int(n_map.get(gene, 0))
        avail = len(group)
        if n_req <= 0:
            # skip entirely
            reports.append({"gene_id": gene, "n_req": n_req, "n_avail": avail, "n_taken": 0, "capped": True})
            continue
        if avail == 0:
            reports.append({"gene_id": gene, "n_req": n_req, "n_avail": 0, "n_taken": 0, "capped": True})
            continue
        if avail <= n_req:
            # take all
            sampled_parts.append(group)
            reports.append({"gene_id": gene, "n_req": n_req, "n_avail": avail, "n_taken": avail, "capped": avail < n_req})
            continue
        # avail > n_req: sample without replacement deterministically
        taken = group.sample(n=n_req, random_state=seed)  # seed provides deterministic
        sampled_parts.append(taken)
        reports.append({"gene_id": gene, "n_req": n_req, "n_avail": avail, "n_taken": n_req, "capped": False})

    if sampled_parts:
        sampled_pdf = pd.concat(sampled_parts, axis=0)
    else:
        sampled_pdf = pd.DataFrame(columns=pdf.columns)

    report_df = pd.DataFrame(reports)
    return pl.from_pandas(sampled_pdf), pl.from_pandas(report_df)

sampled_ds, report = downsample_verbose(null_dedup, real_gene, seed=42)
print("downsampling complete")
print(f"sampled dataset shape: {sampled_ds.shape}")
print(f"genes capped during sampling: {report.filter(pl.col('n_taken') < pl.col('n_req')).height}")

# 9) Downsampling report details
print("\n=== Downsampling Report ===")
print(report.describe())

capped_report = report.filter(pl.col('n_taken') < pl.col('n_req')).sort(['n_req', 'n_avail'], descending=[True, False])
if capped_report.height > 0:
    print(f"\ntop {min(10, capped_report.height)} capped genes (largest requested):")
    print(capped_report.head(10))

# 10) Save debug artifacts
report_path = Path.cwd() / "downsample_debug_report.parquet"
print(f"\nsaving report to {report_path}")
report.write_parquet(report_path, statistics=True)

# 11) Final sanity check: compare requested vs sampled
print("\n=== Final Sanity Check ===")
sampled_counts = sampled_ds.group_by("gene_id").agg(pl.count().alias("n_sampled"))
cmp2 = real_gene.select(["gene_id", "n_variants"]).join(sampled_counts, on="gene_id", how="left").with_columns(pl.col("n_sampled").fill_null(0).cast(pl.Int64))
diffs = cmp2.with_columns((pl.col("n_variants") - pl.col("n_sampled")).alias("diff"))
print("diffs (requested - sampled):")
print(diffs.select(pl.col("diff")).describe())

diff_examples = diffs.filter(pl.col("diff") > 0).sort("diff", descending=True)
if diff_examples.height > 0:
    print(f"\n{diff_examples.height} genes with diff > 0 (top 10):")
    print(diff_examples.head(10).to_pandas())
else:
    print("\nno genes with deficit (all matched perfectly)")

print("\n=== Diagnostics Complete ===")

=== DEDUPLICATION AND DOWNSAMPLING ===
PROJECT ROOT: /Users/markus/in-silico-vg-analysis/02_vg_perm

=== INPUT FILES ===

CLINGEN:
  real_var:  /Users/markus/in-silico-vg-analysis/experiments_data/dataset3_ClinGen/dataset3_clingen_variant_level_summary.parquet
  null_var:  /Users/markus/in-silico-vg-analysis/experiments_data/dataset5_NULL/dataset5_ClinGen_NULL_variant_level_summary_1901.parquet
  real_gene: /Users/markus/in-silico-vg-analysis/experiments_data/dataset3_ClinGen/clingen_genes_20260102.parquet

BACKGROUND:
  real_var:  /Users/markus/in-silico-vg-analysis/experiments_data/dataset4_background/background_variants_20251230.parquet
  null_var:  /Users/markus/in-silico-vg-analysis/experiments_data/dataset5_NULL/dataset5_Background_NULL_variant_level_summary_1901.parquet
  real_gene: /Users/markus/in-silico-vg-analysis/experiments_data/dataset4_background/background_genes_20260102.parquet

PROCESSING: CLINGEN

real genes: 316 genes, 1446682 total variants
mean: 4578.1, median: 44

/var/folders/4k/lmr1c7ss61n50vqkw_40j0pm0000gn/T/ipykernel_64633/4005537317.py:252: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  sampled_counts = sampled_ds.group_by("gene_id").agg(pl.count().alias("n_sampled"))
